# GPT 멀티 펑션콜링 (Responses API, Parallel Function Calling)

OpenAI **Responses API**에서는 모델이 한 번의 응답에 **여러 개의 `function_call` 항목**을 반환할 수 있습니다 (`parallel_tool_calls=True`, 기본 활성화).

핵심 규칙:
- `response.output`에서 `type == "function_call"` 항목들을 골라 실행
- `response.output` 전체를 다음 요청의 `input`에 그대로 이어붙인 뒤, 각 호출에 대해 `function_call_output`(같은 `call_id`)을 추가
- `function_call`이 더 이상 없으면 `response.output_text`가 최종 답변

In [1]:
import json

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # .env 의 OPENAI_API_KEY 로드

client = OpenAI()
MODEL = "gpt-5-nano"

## 1. 도구 정의

Responses API의 함수 도구는 `type: "function"`에 `name`/`parameters`가 **평평하게(flat)** 들어갑니다 (Chat Completions의 `function: {...}` 중첩 구조와 다름).
`strict: True` + `additionalProperties: False`를 쓰면 인자가 스키마를 정확히 따르는 것이 보장됩니다.

In [2]:
TOOLS = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "지정한 도시의 현재 날씨와 기온을 조회한다. 사용자가 날씨나 기온을 물으면 이 도구를 호출한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "도시 이름 (영문), 예: Seoul, Tokyo, Paris"}
            },
            "required": ["city"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "USD 기준 환율을 조회한다. 사용자가 환율이나 통화 변환을 물으면 이 도구를 호출한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "currency": {"type": "string", "description": "통화 코드, 예: KRW, JPY, EUR"}
            },
            "required": ["currency"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

## 2. 도구 구현 (모의 데이터)

Claude 노트북(`notebooks/claude/`)과 동일한 시나리오로, 두 API의 패턴을 비교하기 좋게 맞췄습니다.

In [3]:
MOCK_WEATHER = {
    "Seoul": {"condition": "맑음", "temp_c": 31},
    "Tokyo": {"condition": "흐림", "temp_c": 29},
    "Paris": {"condition": "비", "temp_c": 22},
}
MOCK_RATES = {"KRW": 1385.2, "JPY": 157.8, "EUR": 0.92}


def get_weather(city: str) -> dict:
    if city not in MOCK_WEATHER:
        raise ValueError(f"'{city}' 날씨 정보 없음. 가능한 도시: {list(MOCK_WEATHER)}")
    return {"city": city, **MOCK_WEATHER[city]}


def get_exchange_rate(currency: str) -> dict:
    if currency not in MOCK_RATES:
        raise ValueError(f"'{currency}' 환율 정보 없음. 가능한 통화: {list(MOCK_RATES)}")
    return {"base": "USD", "currency": currency, "rate": MOCK_RATES[currency]}


TOOL_FUNCTIONS = {"get_weather": get_weather, "get_exchange_rate": get_exchange_rate}

## 3. 에이전트 루프

**한 사이클 = `client.responses.create` 1회 = 모델 응답 1건.** `response.output` 전체를 `input`에 이어붙여 히스토리를 유지하고, 그 응답에 담긴 각 `function_call`마다 `function_call_output`(같은 `call_id`)을 추가합니다. **한 응답에 `function_call`이 여러 개면 그게 곧 병렬 함수호출(한 사이클에 N건)** 입니다.
실행이 실패해도 결과를 빼먹지 말고 에러 문자열을 `output`으로 돌려줘야 모델이 스스로 복구할 수 있습니다.

In [4]:
def run_agent(user_message: str) -> str:
    input_list = [{"role": "user", "content": user_message}]
    cycle = 0
    cycle_calls = []  # 사이클별 function_call 개수 (한 사이클 = 모델 응답 1건)

    while True:
        cycle += 1
        response = client.responses.create(
            model=MODEL,
            input=input_list,
            tools=TOOLS,
            parallel_tool_calls=True,  # 기본값이지만 명시
        )
        # function_call 항목을 포함한 출력 전체를 히스토리에 보존
        input_list += response.output
        function_calls = [item for item in response.output if item.type == "function_call"]
        cycle_calls.append(len(function_calls))

        # function_call이 없으면 이 사이클의 응답이 곧 최종 답변
        if not function_calls:
            print(f"═══ 사이클 {cycle} (모델 호출 #{cycle}) ═══  function_call 0개 → 최종 답변")
            print(f"\n요약: 총 {cycle}사이클 · 사이클별 function_call 수 = {cycle_calls}")
            print(f"      → {cycle_calls[0]}건이 '사이클 1의 단일 모델 응답'에서 한꺼번에(병렬) 나옴\n")
            return response.output_text

        # ── 한 번의 모델 응답(= 1 사이클)에서 나온 병렬 function_call을 강조해서 표시 ──
        print(f"═══ 사이클 {cycle} (모델 호출 #{cycle}) ═══")
        print(f"  ⚡ 이 한 번의 응답에서 function_call {len(function_calls)}개가 "
              f"동시에 나왔습니다 (병렬 · parallel_tool_calls)")
        for i, call in enumerate(function_calls, 1):
            print(f"     [{i}/{len(function_calls)}] {call.name}({call.arguments})")

        for call in function_calls:
            try:
                args = json.loads(call.arguments)
                result = json.dumps(TOOL_FUNCTIONS[call.name](**args), ensure_ascii=False)
            except Exception as e:
                result = f"Error: {e}"
            input_list.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": result,
            })
        print(f"  → {len(function_calls)}건 모두 실행 → 결과 {len(function_calls)}개를 "
              f"입력에 추가하고 다음 사이클로\n")

## 4. 실행

서로 독립적인 조회 6건(날씨 3 + 환율 3)이라, **사이클 1의 단일 모델 응답 하나**에서 `function_call` 6개가 **병렬로 한꺼번에** 나옵니다. 그 6건을 실행해 결과를 돌려주면 **사이클 2**에서 모델이 `function_call` 없이 최종 답변을 냅니다.

즉 이 작업은 **총 2사이클**(모델 호출 2회) — `사이클별 function_call 수 = [6, 0]` — 로 끝납니다. 아래 로그의 `═══ 사이클 N ═══` 머리줄로 **"6개가 한 사이클에서 나왔다"** 를 확인하세요.

In [5]:
answer = run_agent(
    "서울, 도쿄, 파리의 현재 날씨를 알려주고, 1 USD가 원화·엔화·유로로 각각 얼마인지도 알려줘."
)
print("\n=== 최종 답변 ===")
print(answer)

═══ 사이클 1 (모델 호출 #1) ═══
  ⚡ 이 한 번의 응답에서 function_call 6개가 동시에 나왔습니다 (병렬 · parallel_tool_calls)
     [1/6] get_weather({"city":"Seoul"})
     [2/6] get_weather({"city":"Tokyo"})
     [3/6] get_weather({"city":"Paris"})
     [4/6] get_exchange_rate({"currency":"KRW"})
     [5/6] get_exchange_rate({"currency":"JPY"})
     [6/6] get_exchange_rate({"currency":"EUR"})
  → 6건 모두 실행 → 결과 6개를 입력에 추가하고 다음 사이클로

═══ 사이클 2 (모델 호출 #2) ═══  function_call 0개 → 최종 답변

요약: 총 2사이클 · 사이클별 function_call 수 = [6, 0]
      → 6건이 '사이클 1의 단일 모델 응답'에서 한꺼번에(병렬) 나옴


=== 최종 답변 ===
다음은 요청하신 정보입니다.

- 서울: 맑음, 기온 31°C
- 도쿄: 흐림, 기온 29°C
- 파리: 비, 기온 22°C

환율 (1 USD당)
- 1 USD = 1,385.20 KRW
- 1 USD = 157.80 JPY
- 1 USD = 0.92 EUR


## 참고

- **Chat Completions와의 차이**: 구 API에서는 `tool_calls` 배열이 assistant 메시지 안에 중첩되고, 결과를 `role: "tool"` 메시지로 반환합니다. Responses API는 `function_call` / `function_call_output`이 최상위 항목으로 평평하게 들어가는 구조입니다.
- **`tool_choice`**: `"auto"`(기본) 외에 `"required"`(반드시 호출), `{"type": "function", "name": "get_weather"}`(특정 도구 강제)를 쓸 수 있습니다.
- **`parallel_tool_calls=False`**: 한 턴에 도구를 하나씩만 호출하게 제한할 수 있습니다.